# Workspace coverage: aggregate report

Positioning performance mapped over the Y-Z workspace, aggregated per
grid point across every comparable sweep on disk. `MATCH_CYCLES`
filters to one sweep depth when wanted.

Protocol: a 3 x 3 grid over the working envelope, 3 cycles (full
sweeps) per run, 3 runs or more. This deviates deliberately from ISO
9283's cube-diagonal poses because the arm is planar; state the
deviation in the thesis.

*Note: the kernel imports `volcaniarm_calibration` through a `.pth` file; restart the kernel after changing the package code. Every figure is also saved to `notebooks/figures/` as a 300 dpi PNG and a vector PDF, ready for the thesis.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    load_runs, select_comparable_runs, filter_runs_by_goals,
    filter_runs_by_cycles, concat_runs, mount_key,
    apply_style, save_fig, run_short, per_axis_residuals_mm,
    PRIMARY, ACCENT, RUN_COLORS, FIG_FULL, FIG_TALL, FIG_SQUARE,
    per_point_accuracy, per_point_repeatability,
    threshold_color,
)
apply_style()

TEST_NAME = 'workspace_coverage'

GROUP = 'pose'           # grids are matched as a whole; leave as 'pose'
                         # 'pose' = same-pose averaging; 'all' = every pose pooled
POSE = None              # (y, z) used by GROUP='pose'; None = pose with most runs
MATCH_CYCLES = None      # e.g. 30 to keep only 30-cycle runs; None = include all
RUN_DIRS = None          # pin the exact run set for final thesis figures
ALLOW_MOUNT_KEYS = None  # merge legacy mount keys known to be identical
MIN_RUNS = 3             # protocol target; fewer runs still compute

all_runs = load_runs(TEST_NAME, run_dirs=RUN_DIRS)
HAVE = bool(all_runs)
if not HAVE:
    print('No completed runs on disk; record some from the calibration '
          'dashboard first.')

if HAVE:
    by_target = {}
    for r in all_runs:
        key = tuple(round(float(v), 3) for v in r['config']['goals'][0])
        by_target.setdefault(key, []).append(r)
    print('Targets on disk:')
    for key, rs in sorted(by_target.items()):
        print(f'  y={key[0]:+.3f} z={key[1]:.3f}: {len(rs)} run(s), '
              f'latest {rs[-1]["config"].get("run_id")}')
    print()

    if MATCH_CYCLES is not None:
        n_before = len(all_runs)
        all_runs = filter_runs_by_cycles(all_runs, MATCH_CYCLES)
        print(f'Cycle filter: kept {len(all_runs)}/{n_before} runs '
              f'with num_cycles == {MATCH_CYCLES}')
        if not all_runs:
            raise RuntimeError('no runs left after the cycle filter')

    if GROUP == 'pose':
        runs = (filter_runs_by_goals(all_runs, [POSE]) if POSE is not None
                else all_runs)
        if not runs:
            raise RuntimeError(f'no completed runs at pose {POSE}; '
                               'see the list above')
        runs = select_comparable_runs(runs, allow_mount_keys=ALLOW_MOUNT_KEYS)
    elif GROUP == 'all':
        runs = select_comparable_runs(all_runs,
                                      allow_mount_keys=ALLOW_MOUNT_KEYS,
                                      match_goals=False)
    else:
        raise ValueError(f"unknown GROUP {GROUP!r}")
    df = concat_runs(runs)
    n_targets = df[['goal_y', 'goal_z']].drop_duplicates().shape[0]
    goal_y, goal_z = runs[-1]['config']['goals'][0]
    run_ids = list(dict.fromkeys(df['run_id']))

    acc = per_point_accuracy(df)
    rep = per_point_repeatability(df)
    print(f'Runs aggregated: {len(runs)}, observations: {len(df)}, '
          f'grid points: {len(acc)}')
    display(acc)

## Accuracy map

Mean residual at each grid point across all runs; colour diverges
about zero, annotation gives the mean in millimetres.

In [ ]:
if HAVE:
    from matplotlib.colors import TwoSlopeNorm

    span = max(1.0, float(np.nanmax(np.abs(acc['mean_mm']))))
    norm = TwoSlopeNorm(vcenter=0.0, vmin=-span, vmax=span)
    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    sc = ax.scatter(acc['goal_y'], acc['goal_z'], c=acc['mean_mm'],
                    cmap='coolwarm', norm=norm, s=550,
                    edgecolor='#666666', linewidth=1.0, zorder=5)
    for _, p in acc.iterrows():
        ax.annotate(f"{p['mean_mm']:+.1f}", (p['goal_y'], p['goal_z']),
                    textcoords='offset points', xytext=(0, 16),
                    ha='center', fontsize=9)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_xlabel('commanded Y  [m]')
    ax.set_ylabel('commanded Z  [m]')
    ax.set_title('Mean accuracy residual across the workspace')
    fig.colorbar(sc, ax=ax, label='mean residual  [mm]')
    save_fig(fig, 'workspace_coverage_aggregate/accuracy_map')

## Commanded vs attained

Mean attained offset per point as an exaggerated arrow (factor in the
title); world residuals mapped into workspace coordinates (Z flips
sign between the frames).

In [ ]:
if HAVE:
    SCALE = 10
    pts = []
    for (py, pz), g in df.groupby(['goal_y', 'goal_z'], sort=True):
        ry, rz = per_axis_residuals_mm(g)
        pts.append((py, pz, np.nanmean(ry) / 1000.0,
                    -np.nanmean(rz) / 1000.0))
    pts = np.array(pts)
    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    ax.plot(pts[:, 0], pts[:, 1], 'o', ms=10, mfc='none', mec=PRIMARY,
            mew=1.6, label='commanded', zorder=4)
    ax.quiver(pts[:, 0], pts[:, 1], pts[:, 2] * SCALE, pts[:, 3] * SCALE,
              angles='xy', scale_units='xy', scale=1.0, width=0.006,
              color='#d04b4b', zorder=5, label='mean error (exaggerated)')
    ax.plot(pts[:, 0] + pts[:, 2] * SCALE, pts[:, 1] + pts[:, 3] * SCALE,
            'x', ms=7, color='#d04b4b', mew=1.6, zorder=6)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_xlabel('workspace Y  [m]')
    ax.set_ylabel('workspace Z  [m]')
    ax.set_title(f'Commanded vs attained positions (errors x{SCALE})')
    ax.legend(loc='best')
    save_fig(fig, 'workspace_coverage_aggregate/error_vectors')

## Repeatability map

ISO 9283 RP per grid point pooled across runs; hollow points have
fewer than two samples. The within-run RP per point (the
ISO-comparable figure) is in the table below.

In [ ]:
if HAVE:
    have_rp = rep[rep['rp_pooled_mm'].notna()]
    missing = rep[rep['rp_pooled_mm'].isna()]
    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    if len(have_rp):
        sc = ax.scatter(have_rp['goal_y'], have_rp['goal_z'],
                        c=have_rp['rp_pooled_mm'], cmap='Blues',
                        vmin=0.0, s=550, edgecolor='#666666',
                        linewidth=1.0, zorder=5)
        for _, p in have_rp.iterrows():
            ax.annotate(f"{p['rp_pooled_mm']:.1f}",
                        (p['goal_y'], p['goal_z']),
                        textcoords='offset points', xytext=(0, 16),
                        ha='center', fontsize=9)
        fig.colorbar(sc, ax=ax, label='pooled RP  [mm]')
    if len(missing):
        ax.plot(missing['goal_y'], missing['goal_z'], 'o', ms=10,
                mfc='none', mec='#aaaaaa', mew=1.4,
                label='fewer than 2 samples')
        ax.legend(loc='best')
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_xlabel('commanded Y  [m]')
    ax.set_ylabel('commanded Z  [m]')
    ax.set_title('Repeatability (RP) across the workspace')
    save_fig(fig, 'workspace_coverage_aggregate/rp_map')
    display(rep)

## Residual profiles along the axes

Mean residual against commanded Y (one line per Z level) and against
commanded Z (one line per Y column). A monotone trend along an axis
points at a specific kinematic parameter.

In [ ]:
if HAVE:
    fig, axes = plt.subplots(1, 2, figsize=(6.3, 3.2), sharey=True)
    for i, (zval, g) in enumerate(acc.groupby('goal_z')):
        g = g.sort_values('goal_y')
        axes[0].plot(g['goal_y'], g['mean_mm'], '-o', ms=6, lw=1.4,
                     color=RUN_COLORS[i % len(RUN_COLORS)],
                     mec='white', mew=0.6, label=f'z = {zval:.3f} m')
    for i, (yval, g) in enumerate(acc.groupby('goal_y')):
        g = g.sort_values('goal_z')
        axes[1].plot(g['goal_z'], g['mean_mm'], '-o', ms=6, lw=1.4,
                     color=RUN_COLORS[i % len(RUN_COLORS)],
                     mec='white', mew=0.6, label=f'y = {yval:.2f} m')
    axes[0].set_xlabel('commanded Y  [m]')
    axes[1].set_xlabel('commanded Z  [m]')
    axes[0].set_ylabel('mean residual  [mm]')
    for ax in axes:
        ax.axhline(0.0, color='#888888', ls='--', lw=1.0)
        ax.legend(loc='best', fontsize=8)
    save_fig(fig, 'workspace_coverage_aggregate/profiles')

## Ranked per-point accuracy

Grid points sorted by residual magnitude with confidence intervals,
coloured by weeding zone.

In [ ]:
if HAVE:
    srt = acc.reindex(acc['mean_mm'].abs().sort_values().index)
    labels = [f"({r['goal_y']:+.2f}, {r['goal_z']:.3f})"
              for _, r in srt.iterrows()]
    colors = [threshold_color(abs(v)) for v in srt['mean_mm']]
    fig, ax = plt.subplots(figsize=(6.3, 0.5 * len(srt) + 1.6))
    ax.barh(range(len(srt)), srt['mean_mm'], xerr=srt['ci95_mm'],
            color=colors, edgecolor='white', height=0.6,
            error_kw={'ecolor': '#555555', 'capsize': 3})
    ax.axvline(0.0, color='#888888', lw=1.0)
    ax.set_yticks(range(len(srt)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('mean residual +/- 95 % CI  [mm]')
    ax.set_title('Grid points ranked by residual magnitude')
    save_fig(fig, 'workspace_coverage_aggregate/ranked_points')

## Sample spread per point

Standard deviation of the raw samples at each point; finer-grained
than RP (no 3-sigma term). A single inflated point flags a
pose-specific problem.

In [ ]:
if HAVE:
    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    sc = ax.scatter(acc['goal_y'], acc['goal_z'], c=acc['std_mm'],
                    cmap='Blues', vmin=0.0, s=550,
                    edgecolor='#666666', linewidth=1.0, zorder=5)
    for _, p in acc.iterrows():
        ax.annotate(f"{p['std_mm']:.2f}", (p['goal_y'], p['goal_z']),
                    textcoords='offset points', xytext=(0, 16),
                    ha='center', fontsize=9)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_xlabel('commanded Y  [m]')
    ax.set_ylabel('commanded Z  [m]')
    ax.set_title('Sample standard deviation across the workspace')
    fig.colorbar(sc, ax=ax, label='std of samples  [mm]')
    save_fig(fig, 'workspace_coverage_aggregate/spread_map')

## Worst points

The three grid points with the largest mean residual magnitude and
the largest RP, for targeted follow-up.

In [ ]:
if HAVE:
    worst_acc = acc.reindex(
        acc['mean_mm'].abs().sort_values(ascending=False).index).head(3)
    worst_rp = rep.sort_values('rp_pooled_mm', ascending=False).head(3)
    print('Largest |mean residual|:')
    display(worst_acc)
    print('Largest pooled RP:')
    display(worst_rp)

## Interpretation

A spatially uniform residual indicates one constant offset (fiducial
mount or camera pose) that a single calibration removes; a smooth
gradient indicates kinematic model divergence growing toward the
workspace edges. The repeatability map is bias-free either way.
Weeding thresholds for reference: 10 mm acceptable, 30 mm marginal.